# Phase 6 training (Colab)

**This notebook decides nothing.** Brief, Section 3: *model logic must never
live only in the notebook.* The channel stack, classes, patch geometry, split
strategy and optimiser settings are all in config; the training loop is
`models/train.py`. This notebook mounts Drive, clones the repo, installs what
Colab lacks, and calls `models.train.compare_models`.

If you find yourself editing a cell to change *how* the model trains, the
change belongs in config or in `models/train.py`, or the run will not be
reproducible from the repository.

---

## Two routes, and what each can answer

**Route A — Landslide4Sense.** Real, independently produced, CC BY 4.0
labelled data. Downloads in this notebook; needs nothing from you but a GPU.
It answers the brief's required **architecture comparison**: plain U-Net
against Attention U-Net on a real labelled task.

> It **cannot** answer the brief's *primary* result. Its masks are binary —
> landslide and not-landslide — so it labels neither water nor terrain shadow.
> The **water-versus-shadow confusion matrix cannot be produced from it at any
> sample size.** Classes 1, 3 and 4 will have zero pixels and read IoU 0.00
> for want of data, not through any failure of the architecture.

**Route B — hand-labelled Himalayan patches.** The only route to the
water-versus-shadow result, because no public dataset labels terrain shadow.
Needs labelling first (`docs/labelling.md`), across **at least 3 areas 10 km
apart**.

Set `ROUTE` below.

**Before running:** `Runtime > Change runtime type > GPU`.

In [ ]:
ROUTE = 'A'   # 'A' = Landslide4Sense (works today) | 'B' = hand-labelled patches

DRIVE_ROOT = '/content/drive/MyDrive/himalayan_dam_monitoring'
REPO = 'https://github.com/USER/himalayan_dam_monitoring.git'   # set this
BRANCH = 'main'

## 1. Mount Drive

Checkpoints and the comparison go to Drive so a disconnected session loses
nothing. Route A's 2.5 GB download stays on Colab's local disk, which is
discarded — there is no reason to spend Drive quota on data that can be
re-fetched.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

OUT_DIR = DRIVE_ROOT + '/outputs/phase6'
import os
os.makedirs(OUT_DIR, exist_ok=True)
print('outputs ->', OUT_DIR)

## 2. Clone or update the repository

The clone is the source of truth. Nothing is pasted into this notebook.

**Re-run this cell after pushing any change.** It does two things that each fix a failure mode where a fix appears not to work:

1. **Clones on the first run, hard-resets to `origin/BRANCH` afterwards.** A plain "clone if missing" leaves a stale checkout, so a pushed fix is silently ignored. The HEAD commit is printed, so you can see what you are actually running.
2. **Drops this project's cached modules.** Python caches imports, so after a pull `from models.train import ...` would hand back the OLD module and the identical error would repeat. Purging them lets the next import read the updated files — **no runtime restart, so you keep the 2.5 GB download.**

If the clone fails, the message distinguishes the two causes git does not: an **empty** repository (nothing pushed yet) and a **private** one (Colab has no credentials).


In [ ]:
# Clone on the first run, PULL on later ones.
#
# A plain "clone if the directory is missing" leaves a stale checkout after
# the first run, so a fix pushed to the repository is silently not picked up
# and the same error repeats -- which looks exactly like the fix not working.
import os, subprocess

CHECKOUT = '/content/himalayan_dam_monitoring'


def sync_repo(path, repo, branch):
    """Clone or fast-forward the checkout, and report what it ended up on."""
    if not os.path.isdir(os.path.join(path, '.git')):
        print('cloning...')
        result = subprocess.run(
            ['git', 'clone', '--branch', branch, repo, path],
            cwd='/content', capture_output=True, text=True,
        )
        if result.returncode:
            # f-strings throughout: a continuation line starting with "%"
            # is ambiguous with an IPython magic, and this is the cell people
            # re-run most.
            raise RuntimeError(
                f'git clone failed (exit {result.returncode}).\n\n'
                f'{result.stderr.strip()}\n\n'
                f'The two usual causes, which git does not distinguish:\n'
                f'  1. the repository is EMPTY -- nothing has been pushed to '
                f'branch {branch!r} yet. Commit and push locally first.\n'
                f'  2. the repository is PRIVATE -- Colab has no credentials. '
                f'Make it public, or put a token in the URL.\n'
            )
    else:
        print('pulling...')
        subprocess.run(['git', 'fetch', '--quiet', 'origin'], cwd=path,
                       check=True)
        reset = subprocess.run(
            ['git', 'reset', '--hard', f'origin/{branch}'],
            cwd=path, capture_output=True, text=True,
        )
        if reset.returncode:
            raise RuntimeError(
                f'could not fast-forward to origin/{branch}:\n{reset.stderr}'
            )
        print(reset.stdout.strip())

    head = subprocess.run(['git', 'log', '-1', '--format=%h %ci %s'],
                          cwd=path, capture_output=True, text=True)
    print('HEAD:', head.stdout.strip())


sync_repo(CHECKOUT, REPO, BRANCH)
def purge_project_modules():
    """Drop this project's modules from sys.modules after a sync.

    Python caches imports, so after a pull a later `from models.train import
    ...` hands back the OLD module and the fix appears not to work. Dropping
    them forces the next import to re-read from disk, which avoids having to
    restart the runtime and lose the 2.5 GB download.

    Only this project's own packages are touched. Purging torch or numpy would
    be slow and can leave two copies of a C extension loaded.
    """
    import sys

    ours = (
        'acquisition', 'analysis', 'app', 'config', 'database', 'detection',
        'models', 'preprocessing', 'reports', 'utils',
    )
    dropped = [
        name for name in list(sys.modules)
        if name.split('.')[0] in ours
    ]
    for name in dropped:
        del sys.modules[name]
    if dropped:
        print(f'dropped {len(dropped)} cached module(s) so the next import '
              f'reads the updated files')
    return dropped


purge_project_modules()
%cd /content/himalayan_dam_monitoring

## 3. Install what Colab lacks

Colab has torch and numpy. `segmentation-models-pytorch` is only needed for
the brief's external baseline (`smp_unet`); the controlled ablation needs
nothing extra. `h5py` reads Landslide4Sense's HDF5.

In [ ]:
!pip -q install segmentation-models-pytorch h5py pyyaml pydantic

import torch
print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU. Runtime > Change runtime type > GPU.')

## 4. Route A — fetch Landslide4Sense

~2.5 GB from Zenodo, then unpacked. Skip this cell for Route B.

**Licence: CC BY 4.0**, read from the Zenodo record itself. Attribution is
required wherever a result from it is reported:

> Ghorbanzadeh, O., Xu, Y., Ghamisi, P., Kopp, M. and Kreil, D. (2022),
> "Landslide4Sense: Reference Benchmark Data and Deep Learning Models for
> Landslide Detection", *IEEE Transactions on Geoscience and Remote Sensing*
> 60, 1-17. Data: https://zenodo.org/records/10463239

Only `TrainData` is labelled — the 245 validation and 800 test patches are
withheld for competition ranking, so our own split comes out of the 3,799
training patches.

In [ ]:
# Plain Python, not shell magics. A magic inside a nested block cannot be
# syntax-checked outside IPython, and this is the cell most likely to be
# re-run after a disconnect.
L4S_DIR = '/content/l4s'
L4S_URL = 'https://zenodo.org/records/10463239/files/TrainData.zip'
L4S_MD5 = '408f09247e3dd256458bc2f6e9edbffb'   # from the Zenodo record


def fetch_landslide4sense(target, url, expected_md5):
    """Download and unpack TrainData, verifying the checksum before use."""
    import hashlib, os, urllib.request, zipfile

    train = os.path.join(target, 'TrainData')
    if os.path.isdir(os.path.join(train, 'img')):
        print('already unpacked at', train)
        return train

    os.makedirs(target, exist_ok=True)
    archive = '/content/TrainData.zip'
    if not os.path.isfile(archive):
        print('downloading ~2.5 GB from Zenodo...')
        with urllib.request.urlopen(url) as response, open(archive, 'wb') as out:
            total = int(response.headers.get('Content-Length') or 0)
            done = 0
            while True:
                chunk = response.read(1 << 20)
                if not chunk:
                    break
                out.write(chunk)
                done += len(chunk)
                if total:
                    print('  %.2f / %.2f GB' % (done / 1e9, total / 1e9), end='\r')
        print()

    print('verifying checksum...')
    digest = hashlib.md5()
    with open(archive, 'rb') as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b''):
            digest.update(chunk)
    if digest.hexdigest() != expected_md5:
        raise RuntimeError(
            'checksum mismatch: got %s, expected %s. The download is corrupt '
            'or the record changed; delete %s and retry rather than training '
            'on it.' % (digest.hexdigest(), expected_md5, archive)
        )
    print('  ok')

    print('unpacking...')
    with zipfile.ZipFile(archive) as archive_file:
        archive_file.extractall(target)
    os.remove(archive)
    return train


if ROUTE == 'A':
    import os
    train_dir = fetch_landslide4sense(L4S_DIR, L4S_URL, L4S_MD5)
    print('labelled patches:', len(os.listdir(os.path.join(train_dir, 'img'))))
else:
    print('Route B: skipping. Patches are expected in Drive.')

## 5. Load the configuration for the chosen route

Route A uses `config/landslide4sense.yaml`, an overlay that declares the
**eleven** channels the dataset can actually supply, a 128 px patch, and an
**event** split. Each of those is a departure from the project defaults, and
the overlay explains the cost of each in comments — read them.

The four channels it drops (`S1_VV`, `S1_VH`, `ILLUMINATION`,
`TERRAIN_SHADOW`) are **removed, not zero-filled**. A zero channel is not a
measurement, and filling one would teach the model that "no data" is a valid
value for real terrain.

In [ ]:
import sys
sys.path.insert(0, '/content/himalayan_dam_monitoring')

from config.loader import load_config, load_config_with_overlay

if ROUTE == 'A':
    cfg = load_config_with_overlay('config/landslide4sense.yaml', load_env=False)
    PATCH_DIR = '/content/patches_l4s'
else:
    cfg = load_config(load_env=False)
    PATCH_DIR = DRIVE_ROOT + '/data/training'

m = cfg.models
print('route         ', ROUTE)
print('channels      ', m.in_channels, m.channels)
print('classes       ', m.num_classes, m.classes)
print('patch         ', m.patch.size_px, 'px at', m.patch.scale_m, 'm')
print('split         ', m.split.strategy)
print('monitored     ', m.training.monitor_metric, '(never overall accuracy)')
print('class_weights ', m.training.class_weights)
print('epochs/batch  ', m.training.epochs, '/', m.training.batch_size)

## 6. Route A — convert to this project's patches

The adapter maps Landslide4Sense's 14 bands onto the configured stack,
derives NDWI/MNDWI/NDVI from the Sentinel-2 bands it holds, and maps its
labels (`0` non-landslide, `1` landslide) onto our classes (`0` background,
`2` landslide) using the mapping declared in config.

It **refuses** any configured channel the dataset cannot supply, and
**refuses** an unmapped label value rather than defaulting it to background.

In [ ]:
if ROUTE == 'A':
    from models.landslide4sense import convert, write_report

    report = convert(cfg, L4S_DIR + '/TrainData', PATCH_DIR)
    write_report(report, OUT_DIR + '/landslide4sense_conversion.json')
    for k, v in report.as_dict().items():
        print(f'{k}: {v}')
else:
    print('Route B: patches are expected to exist already at', PATCH_DIR)

## 7. Inspect the split before spending GPU time

Worth checking every run. If a partition is empty, or the class counts show a
class you expected to be present is absent, fix that before training.

In [ ]:
from models.dataset import PatchDataset, assign_splits, discover_patches, split_summary

refs = discover_patches(PATCH_DIR)
splits = assign_splits(refs, cfg)
print({k: v['patches'] for k, v in split_summary(splits).items()})

for name in ('train', 'val', 'test'):
    if splits[name]:
        print(name, PatchDataset(splits[name], cfg).class_pixel_counts())
    else:
        print(name, 'EMPTY -- metrics on it are undefined, not zero')

## 8. Train both models and compare

The brief's required result. `compare_models` trains each, keeps the
checkpoint best by the monitored metric, and evaluates the **test** split
once at the end — it is not consulted during training.

The default pair is the controlled ablation: `plain_unet` is the Attention
U-Net with the gates removed, identical in depth, widths and upsampling, so a
difference between them is attributable to the gates. Add `MODEL_BASELINE` to
also place them against the brief's external `smp` baseline, which differs in
many ways at once.

In [ ]:
from models.train import MODEL_ATTENTION, MODEL_BASELINE, MODEL_PLAIN, compare_models

report = compare_models(
    cfg,
    PATCH_DIR,
    out_dir=OUT_DIR,
    model_names=(MODEL_PLAIN, MODEL_ATTENTION),   # + MODEL_BASELINE for smp
    device='cuda' if torch.cuda.is_available() else 'cpu',
)
print('wrote', OUT_DIR + '/comparison.json')

## 9. The result

Per-class IoU and the water-versus-shadow block. **Overall accuracy is
absent by design**: background dominates, and a model predicting it
everywhere would score well.

An absent class reports `None`, not `0.0` — the difference between "the model
failed at this" and "there was nothing to learn from".

In [ ]:
import json

print('--- per-class IoU on the held-out test split ---')
for cls, vals in report['comparison_iou']['per_class'].items():
    print(f'  {cls:22s} {vals}')

print()
print('--- macro ---')
print(' ', report['comparison_iou']['macro'])

print()
print('--- water vs shadow ---')
for name, block in report['water_vs_shadow'].items():
    print(f'  {name}: {block}')

print()
print('--- runs ---')
for name, run in report['runs'].items():
    print(f"  {name:16s} best epoch {run['best_epoch']:3d} | "
          f"{run['monitor_metric']}={run['best_monitored']} | "
          f"{run['parameters']:,} params | early-stopped {run['stopped_early']}")

## 10. Read the result honestly

Before quoting any number from this run, check these:

1. **Which classes actually had pixels?** Cell 7 printed the counts. A class
   with zero pixels has an undefined IoU, not a bad one.
2. **On Route A, the water-versus-shadow block is empty by construction.**
   The dataset labels neither class. That result requires hand labels.
3. **Did attention help, and by how much against its cost?** The gates add
   parameters; compare `best_monitored` against `parameters` for both. A small
   gain for a large parameter increase is not an endorsement.
4. **One run is not a result.** The split seed is `models.split.seed`; a
   difference that vanishes when it changes was never real.
5. **No accuracy figure here transfers to the Himalaya.** Landslide4Sense is
   not Himalayan, has no SAR, and no shadow labels.

Record what you ran in `docs/validation.md` §27, including the route, the
class counts and the split.

---

Research prototype output. Not an official warning. Verify with authoritative
sources.

Route A data: Ghorbanzadeh et al. (2022), Landslide4Sense, CC BY 4.0,
https://zenodo.org/records/10463239